# Full-Codebook Coding (v2.1)
LLM coding of YouTube crisis comments with the frozen v2.1 codebook.
Batch-by-batch processing per crisis event, FULL mode.

## Methodology

### Motivation
Passes 1-3 (archived in `legacy/`) explored grief/outrage, basic emotions, and
action demands with pilot prompts. This notebook replaces them with a single unified
pass applying `../prompts/codebook-v2.1.md` through `../prompts/coding-prompt.txt`:
language, emotions, moral appraisal, stance, values, class/identity talk, action
demands, response evaluation, and flags - one JSON object per comment.

### Approach
Chunked LLM calls over `comment_id | text` lines pasted into the assembled prompt
(codebook text + FULL mode + per-phase `CASE CONTEXT` from
`../prompts/phase-context.json`). Output strictly one JSON object per line with
the full v2 schema. `EMERGENT` stays empty in FULL mode; anything the codebook
misses goes in `OTHER_NOTE`.

### Pipeline

| Phase | Step | Description |
|---|---|---|
| **1. Load** | Read micro-sample | Preprocessed CSV -> `n_sample=200` rows |
| **2. Preprocess** | Slim + clean | Keep `comment_id, video_id, text`; `dropna`; `strip` |
| **3. Chunk** | Split rows | `chunk_size=5` -> `id | text` lines per call |
| **4. Call** | LLM coding | `generate_content`, strip fences, JSONL-per-line parse, retry-once |
| **5. Export** | Save + inspect | `sample_full_codebook_results.jsonl` + `.csv` -> merge on `row_id` for QA |

### Cross-Analysis
- *Which moral appraisals (negligence vs intentional vs accusation) drive punitive vs retaliatory demands?*
- *How do class/identity frames interact with outrage and stance?*
- *Do reject-reasons for organizational responses cluster by video type?*

### Input / Output
- **Input:** `01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv`
- **Output:** `03-llm-passes/output/{crisis_name}/full-codebook-coding/sample_full_codebook_results.jsonl` + `.csv` (same basename)


## Setup
Import dependencies: `pandas` for data, `dotenv` + `google.genai` for LLM
calls, `json/re/time/pathlib` for chunking, parsing, and checkpointing.
Later cells (parse, summary, CSV export) reuse these imports — run this
first even when rerunning a single cell in a fresh kernel.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types
import random
from tqdm.auto import tqdm

## Configuration
Micro-sample selector. Papermill injects `crisis_name` / `phase_name` per run.

In [ ]:
# --- papermill parameters (injected by runner) ---
crisis_name = "ateneo-drowning"  # default; overridden by papermill
phase_name = "04-reckoning"  # default; overridden by papermill
n_sample = 200
chunk_size = 5
sample_seed = 42
model = "gemini-3.8-flash"
overwrite = False

In [ ]:
# ── Environment & API Setup ─────────────────────────────────────────────
load_dotenv("../../.env")
GEMMA_API_KEY = os.environ.get("GEMMA_API_KEY", "")

if not GEMMA_API_KEY:
    raise RuntimeError("GEMMA_API_KEY missing. Copy .env.example → .env and fill it.")
print("✅ Gemma API key loaded.")
client = genai.Client(api_key=GEMMA_API_KEY)

# ── Path Configurations ─────────────────────────────────────────────────
INPUT_PATH  = f"../../01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv"
PASS_DIR    = f"../output/{crisis_name}/full-codebook-coding"
OUTPUT_PATH = f"{PASS_DIR}/sample_full_codebook_results.jsonl"
LOG_PATH    = f"{PASS_DIR}/sample_full_codebook_failures.log"
CSV_PATH    = OUTPUT_PATH.replace(".jsonl", ".csv")
# Pre-subfolder location (read-only fallback for existing files)
LEGACY_OUTPUT_PATH = f"../output/{crisis_name}/sample_full_codebook_results.jsonl"
PENDING_PATH = f"{PASS_DIR}/sample_full_codebook_pending.log"
PARTIAL_RESP = OUTPUT_PATH + ".partial.jsonl"
PARTIAL_FAIL = LOG_PATH + ".partial.log"
PARTIAL_PENDING = PENDING_PATH + ".partial.log"
# New pass dirs don't exist until first written; append-mode open() never creates them.
Path(PASS_DIR).mkdir(parents=True, exist_ok=True)

# ── Model & Processing Configurations ───────────────────────────────────
MODEL       = model  # "gemini-3.8-flash" via parameters
TEMPERATURE = 0.0
MAX_TOKENS  = 8192  # full 50-field x 5-row output is ~3-5k tokens; ceiling only
REQUEST_TIMEOUT_MS = 60000  # per-request cap; stalls raise into transient retry
BATCH_SIZE  = chunk_size  # 5 default from parameters
WORKERS     = 5           # concurrent threads (used when threaded runner lands)


## Micro-sample ingest & preprocess
Load `n_sample` rows from `INPUT_PATH`, slim to `comment_id` + `video_id` + `text`,
drop nulls, strip whitespace. Keeps `row_id` for lossless merge at inspect time;
`comment_id` is the `id` sent to the LLM and expected back in each JSON object.


In [ ]:
full_df = pd.read_csv(INPUT_PATH)
print(f"Full phase rows: {len(full_df)}")

sample_df = full_df.head(n_sample).copy()  # deterministic for review; n_sample=200
sample_df = sample_df[["comment_id", "video_id", "text"]]
sample_df = sample_df.dropna()
for _c in ("comment_id", "video_id", "text"):
    sample_df[_c] = sample_df[_c].astype(str).str.strip()
sample_df = sample_df[sample_df["text"] != ""].reset_index(drop=True)
sample_df["row_id"] = sample_df.index
print(f"Sample rows: {len(sample_df)}")
sample_df.head()


## Batch chunking
Slice the sample into 5-row chunks and format each chunk as `comment_id | text`
lines (one per row, newlines flattened) for one LLM call apiece - the
`COMMENTS TO CODE [id | comment text]` format from the coding prompt.


In [ ]:
def _to_line(cid, txt):
    flat = str(txt).replace("\r", " ").replace("\n", " ").strip()
    return f"{cid} | {flat}"

chunks = [sample_df.iloc[i:i + BATCH_SIZE] for i in range(0, len(sample_df), BATCH_SIZE)]
chunk_payloads = ["\n".join(_to_line(r["comment_id"], r["text"]) for _, r in c.iterrows()) for c in chunks]
print(f"{len(chunks)} chunks x ~{BATCH_SIZE} rows | first chunk chars: {len(chunk_payloads[0])}")
print(chunk_payloads[0][:600])


## LLM inference
Assemble the v2 prompt (codebook text + FULL mode + per-phase CASE CONTEXT) and
send one `id | text` chunk per call to the Google AI Studio API.
Sequential calls with a 1s pause; raw responses kept in memory for the parse step.
Outage-resilient: exponential backoff (10/20/40/80s) inside the retry, then the chunk
requeues to the back of the queue with a 60s cooldown (max 3 requeues) instead of being
dropped. After 5 consecutive outage-exhausted chunks a circuit breaker aborts gracefully —
partials survive, so a rerun resumes. Exhausted rows land in the pending log, not failures.


In [ ]:
def _to_line(cid, txt):
    flat = str(txt).replace("\r", " ").replace("\n", " ").strip()
    return f"{cid} | {flat}"

_codebook = Path("../prompts/codebook-v2.1.md").read_text(encoding="utf-8").strip()
_wrapper = Path("../prompts/coding-prompt.txt").read_text(encoding="utf-8").strip()
_phase_ctx = json.loads(Path("../prompts/phase-context.json").read_text(encoding="utf-8"))
_case_context = _phase_ctx.get(crisis_name, {}).get(phase_name, "")
if not _case_context:
    raise RuntimeError(f"No phase context for {crisis_name}/{phase_name} in phase-context.json.")
prompt_template = (
    _wrapper
    .replace("MODE: [PILOT | FULL]", "MODE: FULL")
    .replace("<<<paste full text of the codebook here>>>", _codebook)
    .replace("[The data are made about a Youtube video on ..... No judgments about fault.]", _case_context)
)
for _marker in ("<<<paste", "PILOT | FULL", "....."):
    assert _marker not in prompt_template, f"Prompt assembly missed placeholder: {_marker}"
print(prompt_template[:400])

# ── Outage-resilient retry: exponential backoff, requeue, circuit breaker ──
TRANSIENT_TRIES = 5       # 5 attempts => transient sleeps ~10/20/40/80s
TRANSIENT_BASE_S = 10
BACKOFF_CAP_S = 300
RATE_LIMIT_SLEEP = (60, 90)
COOLDOWN_S = 60           # pause before a requeued chunk is attempted again
MAX_REVISITS = 3          # requeues per chunk before it is marked pending
BREAKER_TRIP = 5          # consecutive outage-exhausted chunks before abort

def is_rate_limited(e: Exception) -> bool:
    msg = repr(e)
    return ("429" in msg) or ("RESOURCE_EXHAUSTED" in msg)

def is_transient(e: Exception) -> bool:
    msg = repr(e)
    return any(k in msg for k in ("500", "INTERNAL", "503", "UNAVAILABLE", "HIGH DEMAND",
                                  "Timeout", "TIMEOUT", "TimeoutError", "timed out"))

def call_with_retry(full_input, tries=TRANSIENT_TRIES):
    for attempt in range(tries):
        try:
            return client.models.generate_content(model=MODEL, contents=full_input,
                config=types.GenerateContentConfig(temperature=TEMPERATURE, max_output_tokens=MAX_TOKENS,
                    http_options=types.HttpOptions(timeout=REQUEST_TIMEOUT_MS)))
        except Exception as e:
            if attempt == tries - 1:
                raise
            if is_rate_limited(e):
                sleep_time = random.uniform(*RATE_LIMIT_SLEEP)
            elif is_transient(e):
                sleep_time = min(TRANSIENT_BASE_S * (2 ** attempt) + random.uniform(0, 5), BACKOFF_CAP_S)
            else:
                sleep_time = (2 ** attempt) + random.uniform(0, 1)
            print(f"  [Attempt {attempt+1} failed: {repr(e)}] Retrying in {sleep_time:.1f}s...")
            time.sleep(sleep_time)

chunks_queue = [sample_df.iloc[i:i + BATCH_SIZE] for i in range(0, len(sample_df), BATCH_SIZE)]
raw_responses, hard_failures = [], []
revisit_counts = {}
outage_streak = 0
# Resume-skip: reload succeeded row_ids from partials when not overwriting
# Prior api_failures are retried (not skipped) — only ok_ids filter the queue.
if not overwrite:
    try:
        ok_ids = set()
        if Path(PARTIAL_RESP).exists():
            with open(PARTIAL_RESP, encoding="utf-8") as f:
                for line in f:
                    try:
                        r = json.loads(line)
                        ok_ids.update(r.get("row_ids", []))
                        raw_responses.append(r)
                    except Exception:
                        pass
        prior_failures = []
        if Path(PARTIAL_FAIL).exists():
            with open(PARTIAL_FAIL, encoding="utf-8") as f:
                for line in f:
                    try:
                        e = json.loads(line)
                        if "row_id" in e:
                            prior_failures.append(e)
                    except Exception:
                        pass
            # Clear stale failure checkpoint so a retry appends cleanly instead of duplicating.
            try:
                Path(PARTIAL_FAIL).unlink()
            except Exception:
                pass
        pending_prior = 0
        if Path(PARTIAL_PENDING).exists():
            with open(PARTIAL_PENDING, encoding="utf-8") as f:
                for line in f:
                    try:
                        e = json.loads(line)
                        if "row_id" in e:
                            pending_prior += 1
                    except Exception:
                        pass
        hard_failures = []
        if ok_ids:
            chunks_queue = [c for c in chunks_queue if not set(c["row_id"].tolist()) <= ok_ids]
            print(f"Resuming: {len(ok_ids)} rows done, {len(prior_failures)} prior failures to retry, "
                  f"{pending_prior} pending-transient, {len(chunks_queue)} chunks left.")
    except Exception as e:
        print(f"Resume check failed ({e!r}); starting fresh.")
else:
    for pth in (PARTIAL_RESP, PARTIAL_FAIL, PARTIAL_PENDING):
        try:
            Path(pth).unlink(missing_ok=True)
        except Exception:
            pass
print(f"Starting dynamic queue with {len(chunks_queue)} initial chunks...")

def _append_pending(chunk, reason, error):
    for _, row in chunk.iterrows():
        ent = {"row_id": int(row["row_id"]), "comment_id": row.get("comment_id"),
               "reason": reason, "error": error, "text_length": len(str(row["text"]))}
        with open(PARTIAL_PENDING, "a", encoding="utf-8") as f:
            f.write(json.dumps({"kind": "pending_transient", **ent}, ensure_ascii=False) + "\n")

# Row-level progress (split-safe: total fixed, update on terminal rows only)
total_rows = len(sample_df)
_resumed = len(ok_ids) if (not overwrite and 'ok_ids' in locals()) else 0
pbar = tqdm(total=total_rows, initial=_resumed,
            desc=f"LLM {crisis_name}/{phase_name}", unit="rows")

try:
    while chunks_queue:
        current_chunk = chunks_queue.pop(0)
        n_rows = len(current_chunk)
        payload = "\n".join(_to_line(r["comment_id"], r["text"]) for _, r in current_chunk.iterrows())
        try:
            resp = call_with_retry(prompt_template + "\n\nCOMMENTS TO CODE\n" + payload)
            rec = {"row_ids": current_chunk["row_id"].tolist(), "text": resp.text or ""}
            raw_responses.append(rec)
            with open(PARTIAL_RESP, "a", encoding="utf-8") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            outage_streak = 0
            pbar.update(n_rows)
            pbar.set_postfix(ok=len(raw_responses), fail=len(hard_failures), left=len(chunks_queue))
            time.sleep(1)
        except Exception as e:
            if is_rate_limited(e) or is_transient(e):
                reason = "rate_limited_exhausted" if is_rate_limited(e) else "transient_exhausted"
                outage_streak += 1
                key = tuple(current_chunk["row_id"].tolist())
                revisit_counts[key] = revisit_counts.get(key, 0) + 1
                if revisit_counts[key] > MAX_REVISITS:
                    print(f"   ↳ {reason} after {MAX_REVISITS} requeues. Marking {n_rows} rows pending.")
                    _append_pending(current_chunk, reason, repr(e))
                    pbar.update(n_rows)
                else:
                    print(f"   ↳ {reason}. Requeueing chunk ({revisit_counts[key]}/{MAX_REVISITS}); "
                          f"cooldown {COOLDOWN_S}s.")
                    time.sleep(COOLDOWN_S)
                    chunks_queue.append(current_chunk)
                pbar.set_postfix(ok=len(raw_responses), fail=len(hard_failures), left=len(chunks_queue))
                if outage_streak >= BREAKER_TRIP:
                    print(f"⚡ Circuit breaker: {outage_streak} consecutive outage-exhausted chunks. "
                          f"Aborting with {len(chunks_queue)} chunks queued — partials saved, rerun to resume.")
                    for c in chunks_queue:
                        _append_pending(c, "breaker_abort", repr(e))
                    break
                continue
            print(f"❌ Chunk failure (Size: {n_rows}): {repr(e)}")
            if n_rows > 1:
                print(f"   ↳ Splitting chunk of {n_rows}...")
                mid = n_rows // 2
                chunks_queue.insert(0, current_chunk.iloc[mid:])
                chunks_queue.insert(0, current_chunk.iloc[:mid])
                # no pbar.update — rows requeued, counted on terminal resolution
                pbar.set_postfix(ok=len(raw_responses), fail=len(hard_failures), left=len(chunks_queue))
            else:
                row = current_chunk.iloc[0]
                print(f"   ↳ Hard failure on row_id {row['row_id']}. Logging and skipping.")
                ent = {"row_id": int(row["row_id"]), "comment_id": row.get("comment_id"),
                       "reason": "content_failure", "error": repr(e), "text_length": len(str(row["text"]))}
                hard_failures.append(ent)
                with open(PARTIAL_FAIL, "a", encoding="utf-8") as f:
                    f.write(json.dumps({"kind": "api_failure", **ent}, ensure_ascii=False) + "\n")
                pbar.update(1)
                pbar.set_postfix(ok=len(raw_responses), fail=len(hard_failures), left=len(chunks_queue))
except (KeyboardInterrupt, SystemExit):
    pbar.close()
    print(f"\nInterrupted: {len(raw_responses)} chunks done, {len(chunks_queue)} left. "
          f"Partials at {PARTIAL_RESP} / {PARTIAL_FAIL} / {PARTIAL_PENDING}.")
    raise

pbar.close()
print(f"\nDone. {len(raw_responses)} ok chunks, {len(hard_failures)} hard failures, "
      f"{len(chunks_queue)} chunks left queued.")


## Clean & parse
Strip markdown fences from each raw response, parse it as JSONL-per-line
(fallback: JSON array), and map parsed objects back to exact sample rows via
the queued `row_ids`. Each object's `id` is cross-checked against its
`comment_id`. API `hard_failures` and unparseable chunks both flush to `LOG_PATH`;
transient-outage rows flush to `PENDING_PATH` (pruned below against successes).


In [ ]:
# --- STANDALONE-BOOTSTRAP (no-op in a full top-to-bottom run) ---
# Lets this cell run without re-calling the LLM (run Setup first for imports).
crisis_name = globals().get("crisis_name", "ateneo-drowning")
phase_name = globals().get("phase_name", "04-reckoning")
n_sample = globals().get("n_sample", 200)
if "OUTPUT_PATH" not in globals() or "LOG_PATH" not in globals() or "PARTIAL_RESP" not in globals():
    PASS_DIR = f"../output/{crisis_name}/full-codebook-coding"
    OUTPUT_PATH = f"{PASS_DIR}/sample_full_codebook_results.jsonl"
    LOG_PATH = f"{PASS_DIR}/sample_full_codebook_failures.log"
    PENDING_PATH = f"{PASS_DIR}/sample_full_codebook_pending.log"
    PARTIAL_RESP = OUTPUT_PATH + ".partial.jsonl"
    PARTIAL_FAIL = LOG_PATH + ".partial.log"
    PARTIAL_PENDING = PENDING_PATH + ".partial.log"
hard_failures = globals().get("hard_failures", [])
if "raw_responses" not in globals() or not raw_responses:
    _partial = Path(PARTIAL_RESP)
    if not _partial.exists():
        raise RuntimeError(
            f"No raw_responses in memory and no partial file at {_partial}. "
            "Run the LLM inference cells first (top-to-bottom)."
        )
    raw_responses = [
        json.loads(ln) for ln in _partial.read_text(encoding="utf-8").splitlines() if ln.strip()
    ]
    if not raw_responses:
        raise RuntimeError(
            f"Partial file at {_partial} exists but yielded 0 rows. "
            "Run the LLM inference cells first (top-to-bottom)."
        )
    print(f"Reloaded {len(raw_responses)} chunks from {_partial} (fresh-kernel resume, no LLM call).")
Path(LOG_PATH).parent.mkdir(parents=True, exist_ok=True)
_total_rows = (
    len(sample_df) if "sample_df" in globals() and sample_df is not None
    else sum(len(item.get("row_ids", [])) for item in raw_responses)
)
# comment_id lookup for the `id` cross-check (empty in fresh-kernel CSV-export use)
_id_of = {}
try:
    if "sample_df" in globals() and sample_df is not None \
            and "comment_id" in sample_df.columns and "row_id" in sample_df.columns:
        _id_of = dict(zip(sample_df["row_id"].tolist(), sample_df["comment_id"].astype(str).tolist()))
except Exception:
    _id_of = {}

def strip_fences(raw: str) -> str:
    s = raw.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)
    return s.strip()

def load_objs(txt: str):
    # Coding prompt specifies JSONL (one object per line); accept a JSON array too.
    s = strip_fences(txt)
    if "\n" in s:
        try:
            return [json.loads(ln) for ln in s.splitlines() if ln.strip()]
        except Exception:
            pass
    objs = json.loads(s)
    return objs

REQUIRED_KEYS = {"id","LANGUAGE","ANGER","CONTEMPT","DISGUST","SADNESS","CONDOLENCE","COMPASSION_VICTIMS","SYMPATHY_OFFENDER","FEAR","NORM_VIOLATION","ATTRIBUTION","IGNORED_WARNING","RECURRENCE","COVER_UP","ALT_ACCOUNT","INJUSTICE","SELF_INTEREST","BLAME_TARGETS","STANCE","NEEDS_CONTEXT","VALUES","CLASS_FRAME","CLASS_DIRECTION","IDENTITY_FRAME","APPEARANCE_ATTACK","ACTIONS","RESPONSE_TARGET","RESPONSE_EVAL","REJECT_REASONS","TARGET_OF_COMMENT","META_DISCOURSE","SARCASM","UNCODABLE","evidence","evidence_en","EMERGENT","OTHER_NOTE","confidence"}

parsed_records, parse_failures = [], []
for item in raw_responses:
    row_ids, txt = item["row_ids"], item["text"]
    if not txt.strip():
        parse_failures.append({"row_ids": row_ids, "error": "empty_response"})
        continue
    try:
        objs = load_objs(txt)
    except Exception as e:
        parse_failures.append({"row_ids": row_ids, "error": f"json_decode: {e!r}"})
        continue
    if isinstance(objs, dict):
        objs = [objs]
    if len(objs) != len(row_ids):
        parse_failures.append({"row_ids": row_ids, "error": f"count_mismatch: {len(objs)} objs for {len(row_ids)} rows"})
    for rid, o in zip(row_ids, objs):
        missing = REQUIRED_KEYS - set(o.keys())
        if missing:
            parse_failures.append({"row_ids": [rid], "error": f"missing_keys: {sorted(missing)}"})
            continue
        if _id_of and str(o.get("id", "")).strip() != str(_id_of.get(rid, "")).strip():
            parse_failures.append({"row_ids": [rid], "error": f"id_mismatch: {o.get('id')!r} vs {_id_of.get(rid)!r}"})
            continue
        o["_row_id"] = rid
        parsed_records.append(o)

with open(LOG_PATH, "w", encoding="utf-8") as f:
    for x in hard_failures:
        f.write(json.dumps({"kind": "api_failure", **x}, ensure_ascii=False) + "\n")
    for x in parse_failures:
        f.write(json.dumps({"kind": "parse_failure", **x}, ensure_ascii=False) + "\n")

# Final pending log: drop rows that later succeeded (self-heals stale entries).
_ok_row_ids = {rid for it in raw_responses for rid in it.get("row_ids", [])}
_pending_entries, _seen = [], set()
if Path(PARTIAL_PENDING).exists():
    with open(PARTIAL_PENDING, encoding="utf-8") as f:
        for line in f:
            try:
                ent = json.loads(line)
            except Exception:
                continue
            rid = ent.get("row_id")
            if rid in _ok_row_ids or rid in _seen:
                continue
            _seen.add(rid)
            _pending_entries.append(ent)
with open(PENDING_PATH, "w", encoding="utf-8") as f:
    for ent in _pending_entries:
        f.write(json.dumps(ent, ensure_ascii=False) + "\n")

_emergent_hit = sum(1 for r in parsed_records if str(r.get("EMERGENT", "")).strip())
print(f"parsed: {len(parsed_records)}/{_total_rows} rows | parse failures: {len(parse_failures)} | "
      f"api failures: {len(hard_failures)} | pending (transient): {len(_pending_entries)} | "
      f"EMERGENT non-empty (FULL-mode warning): {_emergent_hit}")


## Inspect & save
Write parsed records to `OUTPUT_PATH` as JSONL, reload it, and merge back to the
sample for visual QA. Merge key is `_row_id == row_id` (exact, split-safe) —
not `comment_id`, which the LLM echoes back and could mistype (caught as
`id_mismatch` at parse time).


In [ ]:
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    for r in parsed_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Wrote {len(parsed_records)} records → {out_path}")

res_df = pd.read_json(out_path, lines=True) if parsed_records else pd.DataFrame()
if "_row_id" in res_df.columns:
    merged = sample_df.merge(res_df, left_on="row_id", right_on="_row_id", how="left", suffixes=("", "_llm"))
    coverage = merged["_row_id"].notna().mean() if len(merged) else 0.0
    n_matched = int(merged["_row_id"].notna().sum())
else:
    merged = sample_df.copy()
    coverage, n_matched = 0.0, 0
print(f"Coverage: {coverage:.1%} ({n_matched}/{len(merged)} rows matched)")
_show = [c for c in ["row_id", "comment_id", "video_id", "text", "LANGUAGE", "STANCE", "ATTRIBUTION", "ACTIONS"] if c in merged.columns]
display(merged[_show].head(10))
for _c in ["LANGUAGE", "STANCE", "ATTRIBUTION"]:
    if _c in res_df.columns:
        print(f"\n== {_c} ==")
        print(res_df[_c].astype(str).value_counts(dropna=False).head(20).to_string())


## Summary counts (in-memory, no LLM call)
QA gate before CSV export: row count plus distributions of the coded
fields. Uses the in-memory frames from Inspect & save when present;
falls back to `OUTPUT_PATH` on disk for standalone reruns. Coded columns
auto-detected (everything except join/text keys) so the cell stays
schema-agnostic.


In [ ]:
# ── Summary counts: rows + coded-field distributions ──
# Prefers in-memory frames (top-to-bottom run); falls back to OUTPUT_PATH on disk.
# Coded columns auto-detected (everything except join/text keys) so one cell fits any pass.
_SKIP_SUM = {"row_id", "comment_id", "video_id", "text", "id", "_row_id", "row_ids"}
_tbl = None
if "merged" in globals() and merged is not None and len(merged):
    _tbl = merged
elif "res_df" in globals() and res_df is not None and len(res_df):
    _tbl = res_df
else:
    try:
        _tbl = pd.read_json(OUTPUT_PATH, lines=True)
        print(f"(standalone fallback — read {len(_tbl)} records from {OUTPUT_PATH})")
    except Exception as _e:
        print(f"⚠️ No in-memory frames and cannot read {OUTPUT_PATH}: {_e!r}")
        print("Run the Clean & parse + Inspect cells first.")
        _tbl = None
if _tbl is not None:
    # Row count + coverage when the merged join key exists
    if "_row_id" in _tbl.columns:
        _n_ok = int(_tbl["_row_id"].notna().sum())
        print(f"n rows: {len(_tbl)} (matched {_n_ok}, missing {len(_tbl) - _n_ok})")
    else:
        print(f"n rows: {len(_tbl)}")
    # Coded-field summaries; list cols stringified so CSV and memory forms agree
    for _c in [c for c in _tbl.columns if c not in _SKIP_SUM]:
        if _c.endswith("_llm") or _c in ("evidence", "evidence_en", "EMERGENT", "OTHER_NOTE"):
            continue
        _s = _tbl[_c].astype(str) if _tbl[_c].apply(lambda v: isinstance(v, list)).any() else _tbl[_c]
        print(f"\n== {_c} ==")
        print(_s.value_counts(dropna=False).head(20).to_string())


## Export CSV from disk (rerunnable, no LLM call)
Reads the saved JSONL (`OUTPUT_PATH`, else `*.partial.jsonl`, else legacy flat path)
and writes the merged QA table to `CSV_PATH`. Handles both final parsed records
(`_row_id` present) and raw chunk partials (`row_ids` + fenced `text`).
Safe to run standalone after the Setup cell — no API key needed.


In [ ]:
# ── Rerunnable CSV export (reads JSONL from disk; no LLM call) ──────────
# Standalone: run the Setup cell first (imports); no API key or LLM call needed.

# Bootstrap defaults when run standalone (no-op in a full top-to-bottom run)
crisis_name = globals().get("crisis_name", "ateneo-drowning")
phase_name = globals().get("phase_name", "04-reckoning")
n_sample = globals().get("n_sample", 200)
if "OUTPUT_PATH" not in globals():
    PASS_DIR = f"../output/{crisis_name}/full-codebook-coding"
    OUTPUT_PATH = f"{PASS_DIR}/sample_full_codebook_results.jsonl"
    LOG_PATH = f"{PASS_DIR}/sample_full_codebook_failures.log"
    CSV_PATH = OUTPUT_PATH.replace(".jsonl", ".csv")
    LEGACY_OUTPUT_PATH = f"../output/{crisis_name}/sample_full_codebook_results.jsonl"
    PARTIAL_RESP = OUTPUT_PATH + ".partial.jsonl"
    INPUT_PATH = f"../../01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv"

COLUMN_ORDER = ["row_id", "comment_id", "video_id", "text", "id", "LANGUAGE",
    "ANGER", "CONTEMPT", "DISGUST", "SADNESS", "CONDOLENCE", "COMPASSION_VICTIMS",
    "SYMPATHY_OFFENDER", "FEAR", "NORM_VIOLATION", "ATTRIBUTION", "IGNORED_WARNING",
    "RECURRENCE", "COVER_UP", "ALT_ACCOUNT", "INJUSTICE", "SELF_INTEREST",
    "BLAME_TARGETS", "STANCE", "NEEDS_CONTEXT", "VALUES", "CLASS_FRAME",
    "CLASS_DIRECTION", "IDENTITY_FRAME", "APPEARANCE_ATTACK", "ACTIONS",
    "RESPONSE_TARGET", "RESPONSE_EVAL", "REJECT_REASONS", "TARGET_OF_COMMENT",
    "META_DISCOURSE", "SARCASM", "UNCODABLE", "evidence", "evidence_en",
    "EMERGENT", "OTHER_NOTE", "confidence"]

def _strip_fences(raw: str) -> str:
    s = raw.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _load_objs(txt: str):
    s = _strip_fences(txt)
    if "\n" in s:
        try:
            return [json.loads(ln) for ln in s.splitlines() if ln.strip()]
        except Exception:
            pass
    objs = json.loads(s)
    return objs

_REQUIRED = {"id","LANGUAGE","ANGER","CONTEMPT","DISGUST","SADNESS","CONDOLENCE","COMPASSION_VICTIMS","SYMPATHY_OFFENDER","FEAR","NORM_VIOLATION","ATTRIBUTION","IGNORED_WARNING","RECURRENCE","COVER_UP","ALT_ACCOUNT","INJUSTICE","SELF_INTEREST","BLAME_TARGETS","STANCE","NEEDS_CONTEXT","VALUES","CLASS_FRAME","CLASS_DIRECTION","IDENTITY_FRAME","APPEARANCE_ATTACK","ACTIONS","RESPONSE_TARGET","RESPONSE_EVAL","REJECT_REASONS","TARGET_OF_COMMENT","META_DISCOURSE","SARCASM","UNCODABLE","evidence","evidence_en","EMERGENT","OTHER_NOTE","confidence"}

_candidates = [OUTPUT_PATH, PARTIAL_RESP, LEGACY_OUTPUT_PATH, LEGACY_OUTPUT_PATH + ".partial.jsonl"]
SRC_JSONL = next((c for c in _candidates if Path(c).exists()), None)
if SRC_JSONL is None:
    print("⚠️ No JSONL found. Checked:")
    [print(f"   - {c}") for c in _candidates]
    print("Run the LLM inference + Clean & parse cells first.")
else:
    print(f"Reading: {SRC_JSONL}")
    # Rebuild sample deterministically from disk (independent of in-memory sample_df)
    _full = pd.read_csv(INPUT_PATH)
    _sample = _full.head(n_sample)[["comment_id", "video_id", "text"]].copy()
    _sample = _sample.dropna()
    for _c in ("comment_id", "video_id", "text"):
        _sample[_c] = _sample[_c].astype(str).str.strip()
    _sample = _sample[_sample["text"] != ""].reset_index(drop=True)
    _sample["row_id"] = _sample.index
    _id_of = dict(zip(_sample["row_id"].tolist(), _sample["comment_id"].tolist()))

    # Load lines; distinguish parsed records vs raw chunk partials
    _raw_lines = [json.loads(ln) for ln in open(SRC_JSONL, encoding="utf-8") if ln.strip()]
    if _raw_lines and "row_ids" in _raw_lines[0]:
        # Raw chunks → parse exactly like the Clean & parse cell
        _parsed, _n_bad = [], 0
        for item in _raw_lines:
            _rids, _txt = item.get("row_ids", []), item.get("text", "")
            if not str(_txt).strip():
                _n_bad += 1
                continue
            try:
                _objs = _load_objs(_txt)
            except Exception:
                _n_bad += 1
                continue
            if isinstance(_objs, dict):
                _objs = [_objs]
            if len(_objs) != len(_rids):
                _n_bad += 1
            for _rid, _o in zip(_rids, _objs):
                if _REQUIRED - set(_o.keys()):
                    _n_bad += 1
                    continue
                if str(_o.get("id", "")).strip() != str(_id_of.get(_rid, "")).strip():
                    _n_bad += 1
                    continue
                _o["_row_id"] = _rid
                _parsed.append(_o)
        print(f"Parsed {len(_parsed)} records from raw chunks (skipped {_n_bad} bad).")
        res_df = pd.DataFrame(_parsed)
    else:
        res_df = pd.read_json(SRC_JSONL, lines=True) if Path(SRC_JSONL).stat().st_size else pd.DataFrame()

    if "_row_id" in res_df.columns:
        merged_disk = _sample.merge(res_df, left_on="row_id", right_on="_row_id", how="left", suffixes=("", "_llm"))
    else:
        merged_disk = _sample.copy()
    for _c in ("comment_id_llm", "video_id_llm", "text_llm"):
        if _c in merged_disk.columns:
            merged_disk = merged_disk.drop(columns=[_c])
    # Serialize list/dict cols so CSV round-trips via json.loads
    for _lc in [c for c in COLUMN_ORDER if c in merged_disk.columns]:
        if _lc in merged_disk.columns and merged_disk[_lc].apply(lambda v: isinstance(v, (list, dict))).any():
            merged_disk[_lc] = merged_disk[_lc].apply(lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v)
    keep = [c for c in COLUMN_ORDER if c in merged_disk.columns]
    out_csv = Path(CSV_PATH)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    merged_disk.sort_values("row_id")[keep].to_csv(out_csv, index=False)
    _cov = merged_disk["_row_id"].notna().mean() if "_row_id" in merged_disk.columns else 0.0
    print(f"Wrote {len(merged_disk)} rows (coverage {_cov:.1%}) → {out_csv}")
    try:
        display(merged_disk[keep].head(10))
    except NameError:
        print(merged_disk[keep].head(10).to_string())
